# Infaction World Generator v0.11.0\nUnified Place generation and navigation MVP. Source pinned to the same Drive snapshot implementation as GitHub.\n

In [ ]:
import urllib.request, pathlib, subprocess, json, shutil
if not shutil.which('node'):
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','nodejs'],check=True)
base = 'https://raw.githubusercontent.com/playrenathea/rantau/a7a63f6aa9532f39f3d1f356b9cded323d2c6529/'
for filename in ['world-generator.js','place-generator.js']:
    pathlib.Path(filename).write_bytes(urllib.request.urlopen(base + filename).read())
def generate(seed=42):
    return json.loads(subprocess.check_output(['node','-e',f'console.log(JSON.stringify(require("./world-generator.js").generate({int(seed)})))'],text=True))
assert generate(42)['version'] == '0.11.0'


In [ ]:
SEED = 42
world = generate(SEED)
print('Factions:')
for f in world['factions']:
    print(f["name"], '|', ' + '.join(f["types"]))
def show(node, depth=0):
    mark = ' [Hidden]' if node.get('secret') else ''
    scale = f' (Scale {node["scale"]})' if node.get('scale') else ''
    print('  '*depth + node['label'] + scale + mark)
    for child in node['children']:
        show(child, depth+1)
print()
show(world['spatialTree'])


In [ ]:
N = 1000
script = '''const G=require('./world-generator.js');let z=0,l=0,s=0,o=0,h=0,p=0,bad=0;for(let i=0;i<%d;i++){const w=G.generate(i);z+=w.zones.length;l+=w.locations.length;s+=w.sites.length;o+=w.zones.filter(x=>x.region==='Outside City').length;h+=w.places.filter(x=>x.secret).length;p+=w.places.length;const seen=new Set();let invalid=false;function walk(q,parent){if(seen.has(q.id))invalid=true;seen.add(q.id);if(q.scale&&parent?.scale&&q.scale>parent.scale)invalid=true;if(q.region&&parent?.region&&q.region!==parent.region)invalid=true;for(const c of q.children)walk(c,q)}walk(w.spatialTree);if(invalid||seen.size!==w.places.length+2||new Set(w.places.map(q=>q.type)).size!==w.places.length||w.zones.length!==w.targets.zones||w.locations.length!==w.targets.locations||w.zones.length<6||w.zones.length>9||w.locations.length<20||w.locations.length>30||w.sites.some(x=>w.locations.filter(y=>y.parentId===x.id).length<2||!w.locations.some(y=>y.parentId===x.parentId)))bad++}console.log(JSON.stringify({n:%d,zoneMean:z/%d,locationMean:l/%d,siteMean:s/%d,outsideMean:o/%d,hiddenShare:h/p,bad}))''' % (N,N,N,N,N,N)
results = json.loads(subprocess.check_output(['node','-e',script],text=True))
assert results['bad'] == 0
print(json.dumps(results,indent=2))
